In [1]:
import os
import random
import numpy as np
import tensorflow as tf
import rasterio

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
IMG_SIZE = (512, 512)
BATCH_SIZE = 8
EPOCHS = 5
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# BASE_PATH = "/Volumes/Windows8_OS/Dataset/Dataset-OG"
BASE_PATH = r"C:\FinalYear\Dataset-OG"

TRAIN_IMG_DIR = os.path.join(BASE_PATH, "Train", "Images")
TEST_IMG_DIR  = os.path.join(BASE_PATH, "Test", "Images")  # if exists

In [3]:
def load_sar_tiff(path):
    def _read(p):
        with rasterio.open(p.decode()) as src:
            vv = src.read(1).astype(np.float32)
            vh = src.read(2).astype(np.float32)

        vv = np.clip(vv, -35, 5)
        vh = np.clip(vh, -40, 0)

        vv = (vv + 35) / 40
        vh = (vh + 40) / 40

        img = np.stack([vv, vh], axis=-1)
        img = tf.image.resize(img, IMG_SIZE).numpy()
        return img

    img = tf.numpy_function(_read, [path], tf.float32)
    img.set_shape([512, 512, 2])
    return img

In [4]:
def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)

    k = tf.random.uniform([], 0, 4, dtype=tf.int32)
    image = tf.image.rot90(image, k)

    tx = tf.random.uniform([], -0.05, 0.05)
    ty = tf.random.uniform([], -0.05, 0.05)

    image = tf.roll(
        image,
        shift=[
            tf.cast(tx * IMG_SIZE[0], tf.int32),
            tf.cast(ty * IMG_SIZE[1], tf.int32)
        ],
        axis=[0, 1]
    )

    return image, label

In [5]:
def build_balanced_dataset(images_root, target_per_class=1200):
    oil_dir = os.path.join(images_root, "Oil")
    no_oil_dir = os.path.join(images_root, "No_Oil")
    lookalike_dir = os.path.join(images_root, "Lookalike")

    oil_files = sorted([os.path.join(oil_dir, f) for f in os.listdir(oil_dir) if f.endswith(".tif")])
    no_oil_files = sorted([os.path.join(no_oil_dir, f) for f in os.listdir(no_oil_dir) if f.endswith(".tif")])
    lookalike_files = sorted([os.path.join(lookalike_dir, f) for f in os.listdir(lookalike_dir) if f.endswith(".tif")])

    combined_no_oil = no_oil_files + lookalike_files
    random.shuffle(combined_no_oil)

    oil_files = oil_files[:target_per_class]
    combined_no_oil = combined_no_oil[:target_per_class]

    paths = oil_files + combined_no_oil
    labels = [1]*len(oil_files) + [0]*len(combined_no_oil)

    return np.array(paths), np.array(labels)


In [6]:
def build_test_dataset(images_root):
    oil_dir = os.path.join(images_root, "Oil")
    no_oil_dir = os.path.join(images_root, "No_Oil")
    lookalike_dir = os.path.join(images_root, "Lookalike")

    oil_files = [os.path.join(oil_dir, f) for f in os.listdir(oil_dir) if f.endswith(".tif")]
    no_oil_files = [os.path.join(no_oil_dir, f) for f in os.listdir(no_oil_dir) if f.endswith(".tif")]
    lookalike_files = [os.path.join(lookalike_dir, f) for f in os.listdir(lookalike_dir) if f.endswith(".tif")]

    paths = oil_files + no_oil_files + lookalike_files
    labels = (
        [1] * len(oil_files) +
        [0] * len(no_oil_files) +
        [0] * len(lookalike_files)
    )

    return np.array(paths), np.array(labels)

In [7]:
from sklearn.model_selection import train_test_split

all_paths, all_labels = build_balanced_dataset(TRAIN_IMG_DIR)

print("Total balanced samples:", len(all_paths))

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\FinalYear\\Dataset-OG/Train/Images/Oil'

In [8]:
train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths,
    all_labels,
    train_size=0.66,
    stratify=all_labels,
    random_state=SEED
)

print("Train samples:", len(train_paths))
print("Val samples:", len(val_paths))

print("Train Oil:", np.sum(train_labels == 1))
print("Train No_Oil:", np.sum(train_labels == 0))
print("Val Oil:", np.sum(val_labels == 1))
print("Val No_Oil:", np.sum(val_labels == 0))

NameError: name 'all_paths' is not defined

In [9]:
test_paths, test_labels = build_test_dataset(TEST_IMG_DIR)

print("Test samples:", len(test_paths))  # 450
print("Test Oil:", np.sum(test_labels == 1))        # 150
print("Test No_Oil:", np.sum(test_labels == 0))     # 300

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\FinalYear\\Dataset-OG/Test/Images/Oil'

In [10]:
def make_dataset(paths, labels, augment_fn=None, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if shuffle:
        ds = ds.shuffle(
            buffer_size=len(paths),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        lambda x, y: (load_sar_tiff(x), y),
        num_parallel_calls=1  # rasterio-safe
    )

    if augment_fn is not None:
        ds = ds.map(augment_fn, num_parallel_calls=1)

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds


In [11]:
train_ds = make_dataset(
    train_paths,
    train_labels,
    augment_fn=augment,
    shuffle=True
)

val_ds = make_dataset(
    val_paths,
    val_labels,
    augment_fn=None,
    shuffle=False
)

test_ds = make_dataset(
    test_paths,
    test_labels,
    augment_fn=None,
    shuffle=False
)

NameError: name 'train_paths' is not defined

In [13]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

def build_cnn():
    model = Sequential()

    for i in range(8):
        if i == 0:
            model.add(Conv2D(
                filters=32,
                kernel_size=(3, 3),
                activation="relu",
                padding="same",
                input_shape=(512, 512, 2)
            ))
        else:
            model.add(Conv2D(
                filters=32,
                kernel_size=(3, 3),
                activation="relu",
                padding="same"
            ))

        model.add(MaxPooling2D(pool_size=(2, 2)))


    model.add(Flatten())

    model.add(Dense(20, activation="relu"))
    model.add(Dense(20, activation="relu"))

    model.add(Dense(1, activation="sigmoid"))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [19]:
from tensorflow.keras.callbacks import Callback
from tensorflow.keras.callbacks import ModelCheckpoint

class SaveBestWithGraph(Callback):
    def __init__(self):
        self.best_val_acc = 0

    def on_epoch_end(self, epoch, logs=None):
        val_acc = logs.get('val_accuracy')

        if val_acc > self.best_val_acc:
            self.best_val_acc = val_acc

            # Save model
            filename = f"best_model_epoch_{epoch+1}_valacc_{val_acc:.4f}.h5"
            self.model.save(filename)
            print(f"\nSaved: {filename}")

            # Save graph
            plt.figure()
            plt.plot(self.model.history.history['accuracy'], color='red')
            plt.plot(self.model.history.history['val_accuracy'], color='green')
            plt.savefig(f"graph_epoch_{epoch+1}.png")
            plt.close()

In [20]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    filepath="best_model_epoch_{epoch:02d}_valacc_{val_accuracy:.4f}.h5",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

In [21]:
model = build_cnn()
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=50,
    callbacks=[checkpoint, SaveBestWithGraph()]
)

NameError: name 'train_data' is not defined

In [22]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(history.history['accuracy'], label='Train Accuracy', color='red')
plt.plot(history.history['val_accuracy'], label='Val Accuracy', color='green')

plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Training vs Validation Accuracy")

plt.savefig("accuracy_plot.png")   # saves graph
plt.close()

NameError: name 'history' is not defined

<Figure size 640x480 with 0 Axes>

In [23]:
from tensorflow.keras.models import load_model
import glob

# get latest saved best model
model_files = glob.glob("best_model_*.h5")
best_model_path = max(model_files, key=os.path.getctime)

best_model = load_model(best_model_path)
print("Loaded:", best_model_path)

ValueError: max() iterable argument is empty

In [24]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# predictions
y_pred = best_model.predict(test_data)
y_pred_classes = np.argmax(y_pred, axis=1)

# true labels
y_true = test_data.classes  # works for generators

# accuracy
accuracy = np.mean(y_pred_classes == y_true)
print("Best Model Accuracy:", accuracy)

# confusion matrix
cm = confusion_matrix(y_true, y_pred_classes)
print("Confusion Matrix:\n", cm)

# optional detailed report
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred_classes))

NameError: name 'best_model' is not defined